In [1]:
%%capture
!pip install -q "transformers>=4.43.0" datasets accelerate huggingface_hub

In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    import os
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    login(tok); os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
    print("HF login ✓")
except Exception as e:
    print(f"Skipping HF login: {e}")


HF login ✓


In [3]:
# ------------------------------------------------------------------------------
# Cell 3 — Imports & device
# ------------------------------------------------------------------------------
import gc, json, time, warnings
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict, Optional
 
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
 
warnings.filterwarnings("ignore")
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [4]:
# ------------------------------------------------------------------------------
# Cell 4 — Config
# ------------------------------------------------------------------------------
@dataclass
class MEMITConfig:
    model_name        : str       = "meta-llama/Llama-3.2-1B"
    # Mid-range MLP layers for LLaMA-3.2-1B (16 layers, 0-indexed)
    layers            : List[int] = field(default_factory=lambda: [8,9,10,11])
    mlp_module_tmp    : str       = "model.layers.{}.mlp.down_proj"
    layer_module_tmp  : str       = "model.layers.{}"
 
    # Stage-1 z-optimisation  (matched to CounterFact-proven settings)
    v_num_grad_steps  : int   = 100
    v_lr              : float = 0.01    # FIX: was 0.05 → too aggressive for MUSE prompts
    v_weight_decay    : float = 0.05    # FIX: was 0.01 → more regularisation needed
    kl_factor         : float = 0.3
    v_num_prefixes    : int   = 5
 
    # Stage-2 weight update
    mom2_update_weight: float = 1.0
    cov_ridge_eps     : float = 9e-2
 
    # Experiment scale (keep small on T4 to avoid OOM)
    batch_size        : int   = 4
 
CFG = MEMITConfig()
 
# Neutral overwrite target used for unlearning
# (we inject "I don't know" as the replacement fact)
NEUTRAL_TARGET = "I don't know"
 
# Prefix bank for z-target averaging (paper §4.2 Eq.16/19)
PREFIX_BANK = [
    "",
    "As we know, ",
    "It is well established that ",
    "According to available information, ",
    "In fact, ",
]
 
print("Config OK") 

Config OK


In [5]:
# ------------------------------------------------------------------------------
# Cell 5 — Load model
# ------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
tokenizer.pad_token = tokenizer.eos_token
 
# device_map={'':0} forces all layers onto cuda:0  — never use 'auto' on Kaggle
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name, torch_dtype=torch.float16, device_map={"": 0}
)
model.eval()
 
GPU0 = next(iter({p.device for p in model.parameters()}))
assert str(GPU0) == "cuda:0", f"Unexpected device: {GPU0}"
print(f"Model on {GPU0}  |  VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB")
  
 

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model on cuda:0  |  VRAM=2.47 GB


In [6]:
 
# ------------------------------------------------------------------------------
# Cell 6 — MUSE-News dataset loading
# ------------------------------------------------------------------------------
@dataclass
class EditRequest:
    subject    : str
    prompt     : str
    target_new : str          # what we want to inject (NEUTRAL_TARGET for unlearning)
    target_old : str          # the fact we want to erase (original answer)
 
def load_muse_requests(n: int = 20) -> List[EditRequest]:
    """
    Load MUSE-News forget_qa split.
 
    MUSE schema per entry:
        question : str   — e.g. "Who wrote the article about X?"
        answer   : str   — the ground-truth answer we want to erase
 
    Subject extraction strategy (FIX over original heuristic):
    - Use the full question as the prompt (it is already self-contained).
    - Extract subject as the *longest noun-phrase-like span* found in the
      question, falling back to the first 3 content words.
    - This is only used for subj_pos lookup in the hook; MUSE prompts are
      questions so the subject is usually in the middle of the sentence.
    """
    print("Loading MUSE-News / knowmem / forget_qa …")
    ds = load_dataset("muse-bench/MUSE-News", name="knowmem", split="forget_qa")
 
    requests = []
    for entry in ds:
        question = entry.get("question", "").strip().rstrip("?") + "?"
        answer   = entry.get("answer", "").strip()
        if not question or not answer:
            continue
 
        # --- Subject extraction (FIX) ---
        # MUSE questions are of the form "What/Who/Which … <subject> …?"
        # Strategy: strip wh-word prefix, take the next 2-4 words as subject.
        words = question.rstrip("?").split()
        wh_words = {"what","who","which","where","when","how","whose"}
        start = 0
        for i, w in enumerate(words):
            if w.lower().rstrip(",") not in wh_words:
                start = i
                break
        subject_words = words[start : start + 4]
        subject = " ".join(subject_words).strip(".,?")
 
        requests.append(EditRequest(
            subject    = subject,
            prompt     = question,
            target_new = NEUTRAL_TARGET,  # inject neutral response
            target_old = answer,
        ))
        if len(requests) >= n:
            break
 
    print(f"Loaded {len(requests)} forget requests.")
    print(f"  Sample prompt  : {requests[0].prompt}")
    print(f"  Sample answer  : {requests[0].target_old}")
    return requests
 
 
def load_muse_retain(n: int = 20) -> List[EditRequest]:
    """Load retain_qa split to measure specificity (retain quality)."""
    print("Loading MUSE-News / knowmem / retain_qa …")
    ds = load_dataset("muse-bench/MUSE-News", name="knowmem", split="retain_qa")
    requests = []
    for entry in ds:
        question = entry.get("question", "").strip().rstrip("?") + "?"
        answer   = entry.get("answer", "").strip()
        if not question or not answer:
            continue
        words = question.rstrip("?").split()
        wh_words = {"what","who","which","where","when","how","whose"}
        start = 0
        for i, w in enumerate(words):
            if w.lower().rstrip(",") not in wh_words:
                start = i
                break
        subject = " ".join(words[start:start+4]).strip(".,?")
        requests.append(EditRequest(
            subject    = subject,
            prompt     = question,
            target_new = answer,   # retain — answer should still be predicted
            target_old = answer,
        ))
        if len(requests) >= n:
            break
    print(f"Loaded {len(requests)} retain requests.")
    return requests
 
 
forget_requests = load_muse_requests(n=CFG.batch_size * 2)
retain_requests = load_muse_retain(n=CFG.batch_size)
 

Loading MUSE-News / knowmem / forget_qa …
Loaded 8 forget requests.
  Sample prompt  : What year did the Orkney Islands become part of Scotland?
  Sample answer  : 1472
Loading MUSE-News / knowmem / retain_qa …
Loaded 4 retain requests.


In [7]:
# ------------------------------------------------------------------------------
# Cell 7 — Evaluation utilities
# ------------------------------------------------------------------------------
@torch.no_grad()
def token_rank(model, prompt: str, target_token: str) -> int:
    """Return the rank of target_token's first subword at the last prompt position."""
    enc = tokenizer(prompt, return_tensors="pt").to(GPU0)
    logits = model(**enc).logits[0, -1]
    tid = tokenizer(" " + target_token, add_special_tokens=False)["input_ids"]
    if not tid:
        return 9999
    tid = tid[0]
    rank = (logits > logits[tid]).sum().item() + 1
    return int(rank)
 
 
@torch.no_grad()
def target_log_prob(model, prompt: str, target: str) -> float:
    """Mean log-prob of target tokens given prompt."""
    full = prompt + " " + target
    enc  = tokenizer(full, return_tensors="pt").to(GPU0)
    p_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    logits = model(**enc).logits[0]
    lp = torch.log_softmax(logits, dim=-1)
    tids = enc["input_ids"][0, p_len:]
    if len(tids) == 0:
        return -999.0
    score = sum(lp[p_len - 1 + i, tid].item() for i, tid in enumerate(tids))
    return score / len(tids)
 
 
def evaluate(
    model,
    forget : List[EditRequest],
    retain : List[EditRequest],
    label  : str,
    pre_forget_lp: Optional[List[float]] = None,
    pre_retain_lp: Optional[List[float]] = None,
):
    """
    Probability-based evaluation (fixes rank>1 metric which was always 1.0).
 
    FQ tracks the mean log-prob of the OLD answer on forget prompts.
      → lower after edit = better unlearning (drop = pre - post, want positive).
    RQ tracks the mean log-prob of the correct answer on retain prompts.
      → should stay close to pre-edit value (drift = post - pre, want ~0).
 
    Also prints rank for interpretability.
    """
    print(f"\n[{label}]")
 
    forget_lps, retain_lps = [], []
 
    for i, r in enumerate(forget):
        lp   = target_log_prob(model, r.prompt, r.target_old)
        enc  = tokenizer(r.prompt, return_tensors="pt").to(GPU0)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1]
        tids = tokenizer(" " + r.target_old, add_special_tokens=False)["input_ids"]
        rank = int((logits > logits[tids[0]]).sum().item() + 1) if tids else 9999
        forget_lps.append(lp)
        drop_str = f"  drop={pre_forget_lp[i] - lp:+.3f}" if pre_forget_lp else ""
        print(f"  FORGET  lp={lp:6.3f}{drop_str}  rank={rank:>5}  '{r.target_old[:25]}'")
 
    for i, r in enumerate(retain):
        lp   = target_log_prob(model, r.prompt, r.target_old)
        enc  = tokenizer(r.prompt, return_tensors="pt").to(GPU0)
        with torch.no_grad():
            logits = model(**enc).logits[0, -1]
        tids = tokenizer(" " + r.target_old, add_special_tokens=False)["input_ids"]
        rank = int((logits > logits[tids[0]]).sum().item() + 1) if tids else 9999
        retain_lps.append(lp)
        drift_str = f"  drift={lp - pre_retain_lp[i]:+.3f}" if pre_retain_lp else ""
        print(f"  RETAIN  lp={lp:6.3f}{drift_str}  rank={rank:>5}  '{r.target_old[:25]}'")
 
    print(f"  Mean forget log-prob : {np.mean(forget_lps):.4f}  (lower after edit = better)")
    print(f"  Mean retain log-prob : {np.mean(retain_lps):.4f}  (stable = less collateral)")
    return forget_lps, retain_lps

In [8]:
# ── Cell 8 — MEMIT core ───────────────────────────────────────────────────────
def get_module(model, name: str):
    m = model
    for p in name.split("."): m = getattr(m, p)
    return m
 
 
def get_last_subject_pos(prompt: str, subject: str) -> int:
    """
    Find the last token position of `subject` inside `prompt`.
    Falls back to second-to-last position if not found.
    """
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0].tolist()
    n = len(prompt_ids)
    for variant in [subject, " " + subject]:
        stoks = tokenizer(variant, add_special_tokens=False)["input_ids"]
        if not stoks:
            continue
        for i in range(n - len(stoks), -1, -1):
            if prompt_ids[i : i + len(stoks)] == stoks:
                return i + len(stoks) - 1
    return max(n - 2, 0)
 
 
def execute_memit_batch(model, tokenizer, batch: List[EditRequest], cfg: MEMITConfig):
    """
    MEMIT batch update.
 
    Stage 1 — z optimisation at the ANCHOR LAYER (cfg.layers[-1]) only.
    ─────────────────────────────────────────────────────────────────────
    z is a single vector in MLP-output / hidden-state space (dim=hidden_dim).
    We hook the down_proj OUTPUT at the subject token in the anchor layer and
    optimise z so the model predicts target_new via:
        loss = NLL(target_new | z injected) + kl_factor * KL(edited || orig)
 
    The same z is then used as the target for ALL layers in Stage 2.
    This follows MEMIT §4.2: one z* found at anchor, then distributed.
 
    Stage 2 — Closed-form ΔW per layer.
    ─────────────────────────────────────
    K = [in_f, n]  down_proj inputs (keys) at subject pos, stacked over batch
    Z = [out_f, n] z targets, stacked over batch
    ΔW = (Z − W₀K) · Kᵀ · (KKᵀ + λI)⁻¹   via linalg.solve (no explicit inv)
    """
    print(f"\nExecuting MEMIT batch (size={len(batch)}) …")
 
    anchor_layer    = cfg.layers[-1]
    anchor_mlp_name = cfg.mlp_module_tmp.format(anchor_layer)
 
    # Probe dims from actual weight tensor
    _w = get_module(model, anchor_mlp_name)
    out_f, in_f = _w.weight.shape   # 2048, 8192 for LLaMA-3.2-1B
 
    z_targets : Dict[int, List[torch.Tensor]] = {l: [] for l in cfg.layers}
    keys       : Dict[int, List[torch.Tensor]] = {l: [] for l in cfg.layers}
 
    # ── Stage 1 ───────────────────────────────────────────────────────────────
    for req_idx, req in enumerate(batch):
        print(f"  [Stage-1] Edit {req_idx+1}/{len(batch)}: '{req.prompt[:55]}'")
 
        target_ids = tokenizer(" " + req.target_new, add_special_tokens=False)["input_ids"]
        target_len = max(len(target_ids), 1)
 
        # z lives in MLP-output space (out_f = hidden_dim = 2048)
        z = torch.nn.Parameter(torch.zeros(out_f, device=GPU0, dtype=torch.float32))
        opt = torch.optim.Adam([z], lr=cfg.v_lr)
 
        prefixes = PREFIX_BANK[: cfg.v_num_prefixes]
 
        # Reference logits for KL penalty (no prefix, full seq, no z injection)
        with torch.no_grad():
            _ref_enc = tokenizer(
                req.prompt + " " + req.target_new, return_tensors="pt"
            ).to(GPU0)
            orig_logits = model(**_ref_enc).logits[0].detach().float()  # [seq, vocab]
 
        for step in range(cfg.v_num_grad_steps):
            opt.zero_grad()
            step_loss = torch.tensor(0.0, device=GPU0)
 
            for prefix in prefixes:
                ctx      = prefix + req.prompt
                full_seq = ctx + " " + req.target_new
                enc      = tokenizer(full_seq, return_tensors="pt").to(GPU0)
                p_len    = tokenizer(ctx, return_tensors="pt")["input_ids"].shape[1]
 
                sp = get_last_subject_pos(ctx, req.subject)
                sp = min(sp, enc["input_ids"].shape[1] - 1)
 
                # Hook: replace down_proj OUTPUT at subject pos in anchor layer only
                _z_ref = z  # capture for closure
                def _z_hook(module, inp, out):
                    out = out.clone()
                    out[0, sp] = _z_ref.to(dtype=out.dtype)
                    return out
 
                anchor_mod = get_module(model, anchor_mlp_name)
                hk = anchor_mod.register_forward_hook(_z_hook)
                output = model(**enc)
                hk.remove()
 
                # NLL on target tokens
                logits_edit = output.logits[0, p_len - 1 : -1].float()
                tgt_ids     = enc["input_ids"][0, p_len:]
                log_p       = torch.log_softmax(logits_edit, dim=-1)
                nll = -sum(log_p[i, tid] for i, tid in enumerate(tgt_ids)) / target_len
 
                # KL penalty — only when seq lengths match (no-prefix case)
                kl_loss = torch.tensor(0.0, device=GPU0)
                if cfg.kl_factor > 0 and output.logits.shape[1] == orig_logits.shape[0]:
                    p_new = torch.softmax(output.logits[0].float(), dim=-1)
                    p_old = torch.softmax(orig_logits, dim=-1).detach()
                    kl_loss = F.kl_div(
                        torch.log(p_new + 1e-10), p_old,
                        reduction="batchmean", log_target=False
                    )
 
                step_loss = step_loss + nll + cfg.kl_factor * kl_loss
 
            (step_loss / len(prefixes)).backward()
            opt.step()
 
        # Same z for all layers (MEMIT distributes one target across layers)
        with torch.no_grad():
            z_final = z.detach().float().clone()
            for layer in cfg.layers:
                z_targets[layer].append(z_final)
 
        # ── Capture key vectors (down_proj inputs) per layer ──────────────────
        k_temp: Dict[int, Optional[torch.Tensor]] = {l: None for l in cfg.layers}
 
        sp_clean  = get_last_subject_pos(req.prompt, req.subject)
        enc_clean = tokenizer(req.prompt, return_tensors="pt").to(GPU0)
        sp_clean  = min(sp_clean, enc_clean["input_ids"].shape[1] - 1)
 
        pre_hooks = []
        for layer in cfg.layers:
            def make_k_hook(l):
                # register_forward_pre_hook: hook_fn(module, args) — 2 args only
                # args[0] is the input tensor to down_proj: [1, seq_len, in_f=8192]
                def hook_fn(module, args):
                    k_temp[l] = args[0][0, sp_clean].detach().float().clone()
                return hook_fn
            mod = get_module(model, cfg.mlp_module_tmp.format(layer))
            pre_hooks.append(mod.register_forward_pre_hook(make_k_hook(layer)))
 
        with torch.no_grad():
            model(**enc_clean)
        for h in pre_hooks: h.remove()
 
        for layer in cfg.layers:
            assert k_temp[layer] is not None, f"Key not captured for layer {layer}"
            keys[layer].append(k_temp[layer])
 
    # ── Stage 2: ΔW per layer ─────────────────────────────────────────────────
    print("  [Stage-2] Computing ΔW …")
    with torch.no_grad():
        for layer in cfg.layers:
            mlp         = get_module(model, cfg.mlp_module_tmp.format(layer))
            W0          = mlp.weight.data.float()       # [out_f, in_f]
            out_f_l, in_f_l = W0.shape
 
            K = torch.stack(keys[layer], dim=1)         # [in_f, n]
            Z = torch.stack(z_targets[layer], dim=1)    # [out_f, n]
 
            assert K.shape[0] == in_f_l,  f"Layer {layer}: K dim {K.shape[0]} != in_f {in_f_l}"
            assert Z.shape[0] == out_f_l, f"Layer {layer}: Z dim {Z.shape[0]} != out_f {out_f_l}"
 
            residual = Z - W0 @ K                       # [out_f, n]
            KKt      = K @ K.t()                        # [in_f, in_f]
            reg      = cfg.cov_ridge_eps * torch.eye(in_f_l, device=GPU0, dtype=torch.float32)
            X        = torch.linalg.solve(KKt + reg, K) # [in_f, n]
            dW       = residual @ X.t()                 # [out_f, in_f]
 
            mlp.weight.data.add_((dW * cfg.mom2_update_weight).half())
            print(f"    Layer {layer:2d}: ΔW norm = {dW.norm().item():.4f}")
 
    print("  Batch update complete ✓")
    gc.collect(); torch.cuda.empty_cache()
 

In [9]:
# ── Cell 10 — Run experiment ──────────────────────────────────────────────────
eval_forget = forget_requests[: CFG.batch_size]
eval_retain = retain_requests[: CFG.batch_size]
edit_batch  = forget_requests[: CFG.batch_size]
 
print("=" * 60)
print("Step 1 — Pre-edit evaluation")
print("=" * 60)
pre_f_lps, pre_r_lps = evaluate(model, eval_forget, eval_retain, label="PRE-EDIT")
 
print("\n" + "=" * 60)
print("Step 2 — Applying MEMIT edits")
print("=" * 60)
execute_memit_batch(model, tokenizer, edit_batch, CFG)
 
print("\n" + "=" * 60)
print("Step 3 — Post-edit evaluation")
print("=" * 60)
post_f_lps, post_r_lps = evaluate(
    model, eval_forget, eval_retain, label="POST-EDIT",
    pre_forget_lp=pre_f_lps, pre_retain_lp=pre_r_lps,
)
 
print("\n" + "=" * 60)
forget_drop  = float(np.mean([p - q for p, q in zip(pre_f_lps,  post_f_lps)]))
retain_drift = float(np.mean([abs(q - p) for p, q in zip(pre_r_lps, post_r_lps)]))
print(f"  Forget log-prob drop  (↑ better) : {forget_drop:+.4f}")
print(f"  Retain log-prob drift (↓ better) : {retain_drift:+.4f}")
print("=" * 60)

Step 1 — Pre-edit evaluation

[PRE-EDIT]
  FORGET  lp=-3.170  rank=    6  '1472'
  FORGET  lp=-2.387  rank=    3  '2005'
  FORGET  lp=-2.315  rank=    7  '1960s'
  FORGET  lp=-3.193  rank=    5  '19%'
  RETAIN  lp=-1.919  rank=    7  '10 women and 10 men'
  RETAIN  lp=-1.669  rank=    5  '1988'
  RETAIN  lp=-2.264  rank=    2  '10%'
  RETAIN  lp=-2.280  rank=    4  '10 million'
  Mean forget log-prob : -2.7664  (lower after edit = better)
  Mean retain log-prob : -2.0328  (stable = less collateral)

Step 2 — Applying MEMIT edits

Executing MEMIT batch (size=4) …
  [Stage-1] Edit 1/4: 'What year did the Orkney Islands become part of Scotlan'
  [Stage-1] Edit 2/4: 'According to Sir Elton, what year did Paul O'Grady host'
  [Stage-1] Edit 3/4: 'In what year did Willie Nelson start out as a songwrite'
  [Stage-1] Edit 4/4: 'What percentage did the AfD party reach in the latest A'
  [Stage-2] Computing ΔW …
    Layer  8: ΔW norm = 4.6128
    Layer  9: ΔW norm = 3.8884
    Layer 10: ΔW norm 